In [4]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

# ==========================================
# STEP 1: SETUP DEVICE & DIRECTORIES
# ==========================================
# Automatically use GPU if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Pointing to the WAVE directories
train_dir = '../data/wave/training'
test_dir  = '../data/wave/testing'

# ==========================================
# STEP 2 & 3: DATA AUGMENTATION & NORMALIZATION
# ==========================================
# PyTorch handles augmentation and normalization simultaneously.
# CRITICAL UPDATE: ResNet expects 224x224 images and specific ImageNet color normalization.

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),                                 # Upscale for ResNet
    transforms.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.9, 1.1)), # Data Augmentation
    transforms.ToTensor(),                                         # Convert to 0.0-1.0 tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406],               # ImageNet standardization
                         std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225])
])

# ==========================================
# STEP 4: LOAD DATASETS
# ==========================================
train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transform)
test_dataset = datasets.ImageFolder(root=test_dir, transform=test_transform)

# DataLoaders handle batching and shuffling
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(f"Training batches: {len(train_loader)} | Testing batches: {len(test_loader)}")

# ==========================================
# STEP 5: BUILD THE RESNET18 TRANSFER LEARNING MODEL
# ==========================================
print("\nLoading Pre-trained ResNet18...")
# Load the pre-trained ResNet18 model
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze the core feature-extraction layers (we don't want to destroy its existing knowledge)
for param in model.parameters():
    param.requires_grad = False

# Replace the final classification head for our specific binary task (Healthy vs Parkinson's)
num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(num_ftrs, 64),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(64, 1),
    nn.Sigmoid()  # Outputs a probability between 0 and 1
)

# Move the finalized model to the CPU/GPU
model = model.to(device)

# ==========================================
# STEP 6: COMPILE (Optimizer and Loss)
# ==========================================
criterion = nn.BCELoss() # Binary Cross Entropy for binary classification

# We only pass the parameters of the NEW head to the optimizer, since the rest is frozen
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# ==========================================
# STEP 7 & 8: TRAINING WITH "LOWEST LOSS" SAVING
# ==========================================
epochs = 15
best_val_loss = float('inf')  # Start with infinite loss and hunt for the lowest
best_acc_at_lowest_loss = 0.0 # Variable to remember the accuracy at that lowest loss

print('\n--- Starting Transfer Learning (15 epochs) ---')

for epoch in range(epochs):
    # --- TRAINING PHASE ---
    model.train() 
    running_loss, correct_train, total_train = 0.0, 0, 0
    
    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.float().unsqueeze(1).to(device)
        
        optimizer.zero_grad()             # Clear old gradients
        outputs = model(inputs)           # Forward pass
        loss = criterion(outputs, labels) # Calculate error
        loss.backward()                   # Backward pass
        optimizer.step()                  # Update weights
        
        running_loss += loss.item() * inputs.size(0)
        predicted = (outputs > 0.5).float()
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
        
    train_loss = running_loss / total_train
    train_acc = correct_train / total_train
    
    # --- VALIDATION PHASE ---
    model.eval() 
    val_loss, correct_val, total_val = 0.0, 0, 0
    
    with torch.no_grad(): # Turn off gradients for testing
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.float().unsqueeze(1).to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            predicted = (outputs > 0.5).float()
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            
    val_loss = val_loss / total_val
    val_acc = correct_val / total_val
    
    print(f"Epoch {epoch+1:02d}/{epochs} - loss: {train_loss:.4f} - acc: {train_acc:.4f} - val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}")
        
    # --- SAVE THE MOST CONFIDENT MODEL AUTOMATICALLY ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_acc_at_lowest_loss = val_acc
        torch.save(model.state_dict(), '../backend/models/wave_resnet18.pth')
        print(f"  🌟 New most confident model! Lowest loss hit: {val_loss:.4f} (Accuracy: {val_acc*100:.2f}%). Saved.")

print(f"\nTraining complete! The most confident model was securely saved with a loss of {best_val_loss:.4f} and {best_acc_at_lowest_loss*100:.2f}% accuracy.")

Using device: cpu
Training batches: 5 | Testing batches: 2

Loading Pre-trained ResNet18...

--- Starting Transfer Learning (15 epochs) ---
Epoch 01/15 - loss: 0.6996 - acc: 0.5278 - val_loss: 0.6922 - val_acc: 0.5000
  🌟 New most confident model! Lowest loss hit: 0.6922 (Accuracy: 50.00%). Saved.
Epoch 02/15 - loss: 0.6599 - acc: 0.6111 - val_loss: 0.6291 - val_acc: 0.6667
  🌟 New most confident model! Lowest loss hit: 0.6291 (Accuracy: 66.67%). Saved.
Epoch 03/15 - loss: 0.5998 - acc: 0.7917 - val_loss: 0.5904 - val_acc: 0.7000
  🌟 New most confident model! Lowest loss hit: 0.5904 (Accuracy: 70.00%). Saved.
Epoch 04/15 - loss: 0.6183 - acc: 0.6250 - val_loss: 0.5759 - val_acc: 0.6000
  🌟 New most confident model! Lowest loss hit: 0.5759 (Accuracy: 60.00%). Saved.
Epoch 05/15 - loss: 0.5587 - acc: 0.7500 - val_loss: 0.5632 - val_acc: 0.6000
  🌟 New most confident model! Lowest loss hit: 0.5632 (Accuracy: 60.00%). Saved.
Epoch 06/15 - loss: 0.5269 - acc: 0.7639 - val_loss: 0.5858 - val